In [2]:
import requests
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()

APP_KEY = os.getenv("KIS_APP_KEY")
APP_SECRET = os.getenv("KIS_APP_SECRET")
BASE_URL = os.getenv("KIS_BASE_URL")

# 1. 토큰 발급
def get_access_token():
    url = f"{BASE_URL}/oauth2/tokenP"
    headers = {"content-type": "application/json"}
    body = {
        "grant_type": "client_credentials",
        "appkey": APP_KEY,
        "appsecret": APP_SECRET
    }
    res = requests.post(url, headers=headers, json=body)
    res.raise_for_status()
    return res.json()["access_token"]

# 2. 국내주식 현재가 조회
def get_domestic_price(stock_code: str):
    token = get_access_token()

    url = f"{BASE_URL}/uapi/domestic-stock/v1/quotations/inquire-price"
    headers = {
        "authorization": f"Bearer {token}",
        "appkey": APP_KEY,
        "appsecret": APP_SECRET,
        "tr_id": "FHKST01010100"
    }
    params = {
        "fid_cond_mrkt_div_code": "J",  # 주식
        "fid_input_iscd": stock_code   # 종목코드 (005930)
    }

    res = requests.get(url, headers=headers, params=params)
    res.raise_for_status()
    return res.json()

# 3. 실행부
if __name__ == "__main__":
    raw = get_domestic_price("005930")

    print(raw)
    print("=== RAW JSON ===")
    print(raw.keys())

    # KIS 응답 구조 확인
    output = raw.get("output", {})
    print("\n=== OUTPUT KEYS ===")
    print(output.keys())

    # DataFrame으로 변환
    df = pd.DataFrame([output])

    print("\n=== DATAFRAME ===")
    print(df.T)   # 세로로 보는 게 이해하기 좋음


{'output': {'iscd_stat_cls_code': '55', 'marg_rate': '20.00', 'rprs_mrkt_kor_name': 'KOSPI200', 'new_hgpr_lwpr_cls_code': '신고가', 'bstp_kor_isnm': '전기·전자', 'temp_stop_yn': 'N', 'oprc_rang_cont_yn': 'N', 'clpr_rang_cont_yn': 'N', 'crdt_able_yn': 'Y', 'grmn_rate_cls_code': '40', 'elw_pblc_yn': 'Y', 'stck_prpr': '117000', 'prdy_vrss': '5900', 'prdy_vrss_sign': '2', 'prdy_ctrt': '5.31', 'acml_tr_pbmn': '3930507591746', 'acml_vol': '34018174', 'prdy_vrss_vol_rate': '272.30', 'stck_oprc': '112400', 'stck_hgpr': '117000', 'stck_lwpr': '112400', 'stck_mxpr': '144400', 'stck_llam': '77800', 'stck_sdpr': '111100', 'wghn_avrg_stck_prc': '115540.37', 'hts_frgn_ehrt': '52.42', 'frgn_ntby_qty': '11764352', 'pgtr_ntby_qty': '8500299', 'pvt_scnd_dmrs_prc': '112966', 'pvt_frst_dmrs_prc': '112032', 'pvt_pont_val': '111466', 'pvt_frst_dmsp_prc': '110532', 'pvt_scnd_dmsp_prc': '109966', 'dmrs_val': '111750', 'dmsp_val': '110250', 'cpfn': '7780', 'rstc_wdth_prc': '33300', 'stck_fcam': '100', 'stck_sspr': '8

In [4]:
import pandas as pd

df = pd.read_csv("../assets/주식데이터20251228.csv", encoding="cp949")

# 종목코드는 항상 6자리 문자열로
df["종목코드"] = df["종목코드"].astype(str).str.zfill(6)

# 검색용 핵심 컬럼만 추출
df_master = df[
    [
        "종목코드",
        "종목명",
        "시장구분",
        "소속부",
        "종가",
        "대비",
        "등락률",
        "시가",
        "고가",
        "저가",
        "거래량",
        "거래대금",
        "시가총액",
    ]
].copy()

print(df_master.head())


     종목코드     종목명    시장구분    소속부     종가   대비   등락률     시가     고가     저가  \
0  060310      3S  KOSDAQ  벤처기업부   1496  -45 -2.92   1541   1547   1494   
1  095570  AJ네트웍스   KOSPI    NaN   4675   65  1.41   4610   4710   4605   
2  006840   AK홀딩스   KOSPI    NaN   8850   10  0.11   8950   8950   8790   
3  054620     APS  KOSDAQ  중견기업부   4100  240  6.22   3925   4130   3825   
4  265520   AP시스템  KOSDAQ  우량기업부  19490  210  1.09  19280  19630  19280   

      거래량        거래대금          시가총액  
0  190333   288098480   79376323840  
1  154640   719422720  211556648325  
2    3808    33546750  117240914850  
3   60796   244195258   75416306100  
4   62151  1211679470  293340501290  


In [5]:
def find_stock_by_name(df, keyword):
    """
    부분 일치 검색
    """
    result = df[df["종목명"].str.contains(keyword, case=False, na=False)]
    return result


In [7]:
find_stock_by_name(df_master, "한화")
#find_stock_by_name(df_master, "LIG")


,종목코드,종목명,시장구분,소속부,종가,대비,등락률,시가,고가,저가,거래량,거래대금,시가총액
622,003830,대한화섬,KOSPI,NaN,118200,-1500,-1.25,119700,119700,118100,186,22066000,156969600000
2784,000880,한화,KOSPI,NaN,80800,-2600,-3.12,84400,84500,80700,164607,13508464750,6056665788000
2785,00088K,한화3우B,KOSPI,NaN,37000,-100,-0.27,37100,37100,36300,51637,1893657100,717964317000
2786,452260,한화갤러리아,KOSPI,NaN,1366,-44,-3.12,1395,1532,1366,35798860,51988492612,264812227260
2787,45226K,한화갤러리아우,KOSPI,NaN,7250,-510,-6.57,6600,9990,6200,5795593,48508622020,21057407500
2788,451800,한화리츠,KOSPI,NaN,4125,15,0.36,4185,4185,4100,175874,724378755,740850000000
2789,489790,한화비전,KOSPI,NaN,42750,-750,-1.72,42900,44500,41800,1155553,50038210200,2158378672500
2790,088350,한화생명,KOSPI,NaN,3270,-125,-3.68,3395,3410,3245,2333121,7689927547,2840093100000
2791,000370,한화손해보험,KOSPI,NaN,5440,-170,-3.03,5590,5630,5380,473935,2578867825,635059697600
2792,009830,한화솔루션,KOSPI,NaN,26650,-900,-3.27,27650,27750,26650,791087,21288441750,4580936084400
